# Ensemble Model -- XGBoost + LSTM Combination

This notebook combines the LSTM predictions (from `lstm_ensemble.ipynb`) with XGBoost:

1. **Load LSTM predictions** from `lstm_predictions.pkl`
2. **XGBoost walk-forward** on the same expanding-window splits
3. **Ensemble** -- weighted blend of LSTM + XGBoost
4. **Full model comparison** -- Naive vs LSTM vs XGBoost vs Ensemble
5. **Weight sensitivity analysis** -- sweep LSTM/XGBoost weights to find optimal blend

**Input:** `features_price.csv`, `features_nlp.csv`, `lstm_predictions.pkl`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import pickle
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = '..'
RANDOM_STATE = 42
TRAIN_PCT = 0.7
np.random.seed(RANDOM_STATE)

## 1. Load LSTM Predictions & Features

In [ ]:
with open('lstm_predictions.pkl', 'rb') as f:
    saved = pickle.load(f)

lstm_results = saved['lstm_results']
tickers = saved['tickers']
all_feature_cols = saved['all_feature_cols']
config = saved['config']

INITIAL_TRAIN_PCT = TRAIN_PCT
STEP_SIZE = config['STEP_SIZE']

print(f'Loaded LSTM predictions for {len(tickers)} tickers: {tickers}')
print(f'Config: train={INITIAL_TRAIN_PCT}, step={STEP_SIZE}')
for ticker in tickers:
    r = lstm_results[ticker]
    m = r['metrics_lstm']
    print(f'  {ticker}: {len(r["lstm"])} predictions, '
          f'RMSE={m["RMSE"]:.2f}, Dir_Acc={m["Dir_Acc"]:.1f}%')

In [ ]:
price_features = pd.read_csv(f'{DATA_DIR}/features_price.csv', parse_dates=['date'])
nlp_features = pd.read_csv(f'{DATA_DIR}/features_nlp.csv', parse_dates=['date'])

df = price_features.merge(nlp_features, on=['date', 'ticker'], how='left')
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

nlp_cols = [c for c in nlp_features.columns if c not in ['date', 'ticker']]
for col in nlp_cols:
    if col == 'has_news':
        df[col] = df[col].fillna(0).astype(int)
    else:
        df[col] = df[col].fillna(0.0)

df = df.dropna(subset=['target_next_close']).reset_index(drop=True)

print(f'Feature matrix shape: {df.shape}')
print(f'Features used: {len(all_feature_cols)}')

## 2. Helper Functions

In [ ]:
def compute_metrics(actual, predicted):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    actual_dir = np.diff(actual) > 0
    pred_dir = np.diff(predicted) > 0
    dir_acc = np.mean(actual_dir == pred_dir) * 100 if len(actual_dir) > 0 else 0.0
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'Dir_Acc': dir_acc}


def make_xgb_model():
    return xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        min_child_weight=5,
        random_state=RANDOM_STATE, verbosity=0
    )

## 3. XGBoost Walk-Forward + Ensemble

Run XGBoost on the same expanding-window splits used for LSTM, then combine predictions.

In [ ]:
ensemble_results = {}

for ticker in tickers:
    print(f'\n{"=" * 60}')
    print(f'  XGBoost Walk-Forward + Ensemble: {ticker}')
    print(f'{"=" * 60}')

    sub = df[df['ticker'] == ticker].copy().reset_index(drop=True)
    n = len(sub)
    initial_train_size = int(n * INITIAL_TRAIN_PCT)

    features_raw = sub[all_feature_cols].values
    target_raw = sub['target_next_close'].values

    r = lstm_results[ticker]
    lstm_arr = r['lstm']
    naive_arr = r['naive']
    actuals_arr = r['actuals']
    pred_dates = r['dates']

    xgb_predictions = []
    train_end = initial_train_size
    step_num = 0

    while train_end < n:
        predict_end = min(train_end + STEP_SIZE, n)
        step_num += 1

        test_start = train_end
        num_preds = min(STEP_SIZE, n - test_start)

        if num_preds <= 0:
            train_end = predict_end
            continue

        xgb_model = make_xgb_model()
        xgb_model.fit(features_raw[:train_end], target_raw[:train_end])
        xgb_preds = xgb_model.predict(features_raw[test_start:test_start + num_preds])
        xgb_predictions.extend(xgb_preds.tolist())

        if step_num % 4 == 0 or step_num == 1:
            print(f'  Step {step_num}: train={train_end}, predicted {num_preds} days')

        train_end = predict_end

    xgb_arr = np.array(xgb_predictions)

    min_len = min(len(lstm_arr), len(xgb_arr), len(actuals_arr))
    lstm_arr_t = lstm_arr[:min_len]
    xgb_arr_t = xgb_arr[:min_len]
    actuals_arr_t = actuals_arr[:min_len]
    naive_arr_t = naive_arr[:min_len]
    dates_t = pred_dates[:min_len]

    ensemble_arr = 0.6 * lstm_arr_t + 0.4 * xgb_arr_t

    ensemble_results[ticker] = {
        'dates': dates_t,
        'actuals': actuals_arr_t,
        'naive': naive_arr_t,
        'lstm': lstm_arr_t,
        'xgb': xgb_arr_t,
        'ensemble': ensemble_arr,
        'metrics_naive': compute_metrics(actuals_arr_t, naive_arr_t),
        'metrics_lstm': compute_metrics(actuals_arr_t, lstm_arr_t),
        'metrics_xgb': compute_metrics(actuals_arr_t, xgb_arr_t),
        'metrics_ensemble': compute_metrics(actuals_arr_t, ensemble_arr),
    }

    print(f'\n  {ticker} Results ({min_len} prediction days):')
    for name in ['metrics_naive', 'metrics_lstm', 'metrics_xgb', 'metrics_ensemble']:
        m = ensemble_results[ticker][name]
        label = name.replace('metrics_', '').upper()
        print(f'    {label:<10} RMSE={m["RMSE"]:.2f}  MAE={m["MAE"]:.2f}  '
              f'MAPE={m["MAPE"]:.2f}%  Dir_Acc={m["Dir_Acc"]:.1f}%')

print('\n' + '=' * 60)
print('XGBoost + Ensemble completed for all tickers.')

## 4. Full Model Comparison

In [ ]:
full_rows = []

for ticker in tickers:
    r = ensemble_results[ticker]
    for model_name, key in [('Naive', 'metrics_naive'), ('LSTM', 'metrics_lstm'),
                             ('XGBoost', 'metrics_xgb'), ('Ensemble', 'metrics_ensemble')]:
        full_rows.append({'Ticker': ticker, 'Model': model_name, **r[key]})

full_comparison_df = pd.DataFrame(full_rows)

print('=== Mean Performance Across Tickers ===')
summary = full_comparison_df.groupby('Model')[['RMSE', 'MAE', 'MAPE', 'Dir_Acc']].mean().round(2)
summary = summary.reindex(['Naive', 'XGBoost', 'LSTM', 'Ensemble'])
display(summary)

print('\n=== Per-Ticker Breakdown ===')
print(f'{"Ticker":<8} {"Model":<12} {"RMSE":>8} {"MAE":>8} {"MAPE%":>8} {"Dir%":>8}')
print('-' * 54)
for ticker in tickers:
    for model_name, key in [('Naive', 'metrics_naive'), ('LSTM', 'metrics_lstm'),
                             ('XGBoost', 'metrics_xgb'), ('Ensemble', 'metrics_ensemble')]:
        m = ensemble_results[ticker][key]
        print(f'{ticker:<8} {model_name:<12} {m["RMSE"]:>8.2f} {m["MAE"]:>8.2f} '
              f'{m["MAPE"]:>8.2f} {m["Dir_Acc"]:>8.1f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
metrics = ['RMSE', 'MAE', 'MAPE', 'Dir_Acc']
titles = ['RMSE (lower is better)', 'MAE (lower is better)',
          'MAPE % (lower is better)', 'Directional Accuracy % (higher is better)']

for ax, metric, title in zip(axes, metrics, titles):
    pivot = full_comparison_df.pivot(index='Ticker', columns='Model', values=metric)
    pivot = pivot[['Naive', 'XGBoost', 'LSTM', 'Ensemble']]
    pivot.plot(kind='bar', ax=ax, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Ticker')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=0)
    ax.legend(fontsize=8)

plt.suptitle('Full Model Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Predicted vs Actual -- All Models

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    r = ensemble_results[ticker]
    dates = r['dates']

    axes[i].plot(dates, r['actuals'], label='Actual', linewidth=1.8, color='black')
    axes[i].plot(dates, r['naive'], label='Naive', linewidth=1.0, color='gray', alpha=0.6, linestyle='--')
    axes[i].plot(dates, r['lstm'], label='LSTM', linewidth=1.0, color='coral', alpha=0.8)
    axes[i].plot(dates, r['xgb'], label='XGBoost', linewidth=1.0, color='steelblue', alpha=0.8)
    axes[i].plot(dates, r['ensemble'], label='Ensemble', linewidth=1.2, color='green', alpha=0.9)

    axes[i].set_title(ticker, fontsize=13, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Close ($)')
    if i == 0:
        axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
fig.suptitle('Predicted vs Actual -- All Models', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Ensemble Weight Sensitivity Analysis

Sweep LSTM/XGBoost weight combinations to find the optimal blend.

In [ ]:
weight_results = []

for lstm_weight in np.arange(0.0, 1.05, 0.1):
    xgb_weight = 1.0 - lstm_weight
    ticker_rmses = []
    ticker_dir_accs = []

    for ticker in tickers:
        r = ensemble_results[ticker]
        blended = lstm_weight * r['lstm'] + xgb_weight * r['xgb']
        m = compute_metrics(r['actuals'], blended)
        ticker_rmses.append(m['RMSE'])
        ticker_dir_accs.append(m['Dir_Acc'])

    weight_results.append({
        'LSTM_weight': round(lstm_weight, 1),
        'XGB_weight': round(xgb_weight, 1),
        'Mean_RMSE': np.mean(ticker_rmses),
        'Mean_Dir_Acc': np.mean(ticker_dir_accs)
    })

weight_df = pd.DataFrame(weight_results)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(weight_df['LSTM_weight'], weight_df['Mean_RMSE'], 'bo-', linewidth=2)
best_rmse_idx = weight_df['Mean_RMSE'].idxmin()
axes[0].axvline(weight_df.loc[best_rmse_idx, 'LSTM_weight'], color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('LSTM Weight')
axes[0].set_ylabel('Mean RMSE')
axes[0].set_title(f'Optimal LSTM weight for RMSE: {weight_df.loc[best_rmse_idx, "LSTM_weight"]}',
                  fontsize=12, fontweight='bold')

axes[1].plot(weight_df['LSTM_weight'], weight_df['Mean_Dir_Acc'], 'go-', linewidth=2)
best_dir_idx = weight_df['Mean_Dir_Acc'].idxmax()
axes[1].axvline(weight_df.loc[best_dir_idx, 'LSTM_weight'], color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('LSTM Weight')
axes[1].set_ylabel('Mean Dir. Accuracy (%)')
axes[1].set_title(f'Optimal LSTM weight for Dir. Acc: {weight_df.loc[best_dir_idx, "LSTM_weight"]}',
                  fontsize=12, fontweight='bold')

plt.suptitle('Ensemble Weight Sensitivity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

display(weight_df.round(2))

## Conclusions

### Pipeline Summary

This notebook combined the LSTM predictions (pre-computed in `lstm_ensemble.ipynb`) with XGBoost walk-forward predictions:

- **XGBoost**: Same hyperparameters as the `modeling.ipynb` notebook, same expanding-window splits
- **Ensemble**: Weighted blend of LSTM (0.6) + XGBoost (0.4), with sensitivity analysis

### Key Findings

**Examine the results above (Section 4) to determine:**

- Which standalone model performs better (LSTM vs XGBoost)
- Whether the ensemble outperforms both individual models
- The optimal LSTM/XGBoost weight ratio from the sensitivity sweep (Section 6)
- Whether directional accuracy benefits from the ensemble approach

### SOTA Techniques Applied

| Technique | Source | Implementation |
|-----------|--------|----------------|
| 0.6/0.4 ensemble | Notebook 4 (Netflix) | LSTM-weighted blend with XGBoost |
| Weight sensitivity | Our design | Sweep 0.0-1.0 to find optimal ratio |
| Walk-forward | Our design | Expanding window, retrain every 5 days |